# 🩺 MedGemma server for Aura — Google Colab (free GPU)

Serves **MedGemma 1.5 4B (vision)** via llama.cpp on a free Colab **T4 GPU** and exposes a public OpenAI-compatible endpoint through a free Cloudflare tunnel (no account needed).

### How to use
1. **Runtime → Change runtime type → T4 GPU** (Hardware accelerator).
2. **Runtime → Run all** (first run builds llama.cpp with CUDA — ~10–15 min, one time per session).
3. The last cell prints a line like `MEDGEMMA_ENDPOINT=https://....trycloudflare.com/v1/chat/completions`.
4. Paste that into the app's `.env.local`, set `AI_PROVIDER=openai`, then restart the app and click **AI**.
5. **Keep this tab open** — the link stays alive only while the runtime runs, and a new URL is issued each session.

_Uses the public `unsloth/medgemma-1.5-4b-it-GGUF` — no Hugging Face token or license needed._

In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi -L || echo 'No GPU! Set Runtime > Change runtime type > T4 GPU, then Run all again.'

In [ ]:
# 2) Download the model + vision projector (public repo)
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
REPO = 'unsloth/medgemma-1.5-4b-it-GGUF'
MODEL  = hf_hub_download(REPO, 'medgemma-1.5-4b-it-Q4_K_M.gguf')
MMPROJ = hf_hub_download(REPO, 'mmproj-F16.gguf')
print('model :', MODEL)
print('mmproj:', MMPROJ)

In [ ]:
# 3) Build llama.cpp with CUDA (one time per session, ~10-15 min)
%cd /content
![ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF >/tmp/cmake.log 2>&1
!cmake --build llama.cpp/build --config Release -j 2 --target llama-server 2>&1 | tail -4
!ls -la llama.cpp/build/bin/llama-server && echo 'BUILD OK'

In [ ]:
# 4) Get the Cloudflare tunnel binary (free, no signup)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared && echo 'cloudflared ready'

In [ ]:
# 5) Start MedGemma server (GPU) + public tunnel, then print the endpoint
import subprocess, time, re, requests

srv = subprocess.Popen(
    ['/content/llama.cpp/build/bin/llama-server', '-m', MODEL, '--mmproj', MMPROJ,
     '-ngl', '999', '-c', '4096', '--host', '127.0.0.1', '--port', '8080', '--jinja'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print('Loading MedGemma onto the GPU...')
ready = False
for _ in range(180):
    try:
        if requests.get('http://127.0.0.1:8080/health', timeout=2).ok:
            ready = True; break
    except Exception:
        pass
    time.sleep(2)
print('Server ready ✅' if ready else 'Server slow to start — check logs; the tunnel may still come up.')

cf = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080', '--no-autoupdate'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
for line in cf.stdout:
    m = re.search(r'https://[-a-z0-9.]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
    if time.time() - t0 > 60: break

print('\n' + '=' * 64)
if url:
    print('  Paste this line into the app .env.local (with AI_PROVIDER=openai):\n')
    print('  MEDGEMMA_ENDPOINT=' + url + '/v1/chat/completions')
else:
    print('  Tunnel URL not detected — re-run this cell.')
print('=' * 64)
print('Keep this tab open. The endpoint dies when the runtime stops.')

In [ ]:
# 6) (Optional) keep-alive + quick self-test of the endpoint
import requests, json
r = requests.post('http://127.0.0.1:8080/v1/chat/completions',
                  json={'model': 'medgemma', 'messages': [{'role': 'user', 'content': 'Reply with: MedGemma is online.'}], 'max_tokens': 32})
print(r.json().get('choices', [{}])[0].get('message', {}).get('content', r.text))